In [1]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import os, re, random, pickle, numpy as np
import tensorflow as tf

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# Reproducibility
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

# Paths & Configuration
TXT_PATH = "/content/drive/MyDrive/DLLM/61262-0.txt"
OUT_DIR  = "/content/drive/MyDrive/DLLM"
os.makedirs(OUT_DIR, exist_ok=True)

MODEL_PATH = os.path.join(OUT_DIR, "poirot_model.keras")
TOK_PATH   = os.path.join(OUT_DIR, "tokenizer.pkl")

SEQ_LENGTH = 40
MAX_VOCAB  = 10000
BATCH_SIZE = 128
EPOCHS     = 30

# Load & Trimming Gutenberg

with open(TXT_PATH, "r", encoding="utf-8") as f:
    raw = f.read()

low = raw.lower()

# End Marker
end_candidates = [
    low.find("*** end of this project gutenberg ebook"),
    low.find("*** end of the project gutenberg ebook"),
    low.find("*** end of project gutenberg"),
    low.find("*** end of"),
]
end_candidates = [i for i in end_candidates if i != -1]
end_idx = min(end_candidates) if end_candidates else len(raw)

# Noise Control
start_idx = low.find("the adventure of")
if start_idx == -1:
    start_idx = low.find("i was standing at the window")
if start_idx == -1:
    start_candidates = [
        low.find("poirot investigates"),
        low.find("*** start of this project gutenberg ebook"),
        low.find("*** start of the project gutenberg ebook"),
        low.find("*** start of project gutenberg"),
        low.find("*** start of"),
    ]
    start_candidates = [i for i in start_candidates if i != -1]
    start_idx = min(start_candidates) if start_candidates else 0

text = raw[start_idx:end_idx].lower()

# Normalizing Gutenberg Typography
text = (text
        .replace("’", "'").replace("‘", "'")
        .replace("“", '"').replace("”", '"')
        .replace("—", " - ").replace("–", "-")
)

# Cleaning text
text = re.sub(r"[^a-z0-9\s\.\,\!\?\;\:\'\"\-]", " ", text)

text = re.sub(r"([.,!?;:\"-])", r" \1 ", text)

text = re.sub(r"\s+", " ", text).strip()

print("Sample text:\n", text[:500], "\n")

# Tokenization
tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token="<unk>", filters="")
tokenizer.fit_on_texts([text])

with open(TOK_PATH, "wb") as f:
    pickle.dump(tokenizer, f)
print("Saved tokenizer", TOK_PATH)

sequence = tokenizer.texts_to_sequences([text])[0]
print("Vocab cap:", MAX_VOCAB)
print("First 20 tokens:", sequence[:20])

UNK_ID = tokenizer.word_index.get("<unk>", 1)

# Build Sequences
input_sequences = []
for i in range(SEQ_LENGTH, len(sequence)):
    input_sequences.append(sequence[i-SEQ_LENGTH:i+1])

input_sequences = np.array(input_sequences, dtype=np.int32)
X = input_sequences[:, :-1]
y = input_sequences[:, -1]

print("Total sequences:", input_sequences.shape)
print("X shape:", X.shape, "y shape:", y.shape)

# Model
model = Sequential([
    Input(shape=(SEQ_LENGTH,)),
    Embedding(input_dim=MAX_VOCAB, output_dim=200, mask_zero=True),
    LSTM(256, return_sequences=True),
    Dropout(0.2),
    LSTM(256),
    Dropout(0.2),
    Dense(MAX_VOCAB, activation="softmax"),
])

opt = tf.keras.optimizers.Adam(learning_rate=1e-3)

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=opt,
    metrics=["accuracy"]
)

model.summary()

# Training
callbacks = [
    ModelCheckpoint(MODEL_PATH, save_best_only=True, monitor="val_loss"),
    EarlyStopping(patience=4, restore_best_weights=True, monitor="val_loss"),
]

history = model.fit(
    X, y,
    validation_split=0.1,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    callbacks=callbacks,
    shuffle=True
)

print("Saved best model", MODEL_PATH)

# Reload best model + tokenizer
model = load_model(MODEL_PATH)
with open(TOK_PATH, "rb") as f:
    tokenizer = pickle.load(f)

index_word = tokenizer.index_word
UNK_ID = tokenizer.word_index.get("<unk>", 1)

print("Reloaded best model.")

# Generation Helpers
def normalize_prompt(p: str) -> str:
    p = p.lower().strip()
    p = (p
         .replace("’", "'").replace("‘", "'")
         .replace("“", '"').replace("”", '"')
         .replace("—", " - ").replace("–", "-")
    )
    p = re.sub(r"[^a-z0-9\s\.\,\!\?\;\:\'\"\-]", " ", p)
    p = re.sub(r"([.,!?;:\"-])", r" \1 ", p)
    p = re.sub(r"\s+", " ", p).strip()
    return p

def filter_top_k_top_p(probs, top_k=200, top_p=0.95):
    probs = probs.copy()

    # top-k
    if top_k is not None and top_k > 0:
        kth = np.argpartition(probs, -top_k)[-top_k:]
        mask = np.zeros_like(probs, dtype=bool)
        mask[kth] = True
        probs[~mask] = 0.0

    # top-p
    sorted_idx = np.argsort(probs)[::-1]
    sorted_probs = probs[sorted_idx]
    cumsum = np.cumsum(sorted_probs)

    cutoff = np.searchsorted(cumsum, top_p) + 1
    keep = sorted_idx[:cutoff]

    filtered = np.zeros_like(probs)
    filtered[keep] = probs[keep]
    s = filtered.sum()
    if s <= 0:
        filtered = probs / (probs.sum() + 1e-12)
    else:
        filtered /= s
    return filtered

def generate_poirot_text(prompt, words_to_gen=160, temp=0.9, top_p=0.95, top_k=200):
    # Removing Heavy Penalties that Break Grammar
    generated = normalize_prompt(prompt)

    for i in range(words_to_gen):
        tokens = tokenizer.texts_to_sequences([generated])[0][-SEQ_LENGTH:]

        # RIGHT-padding
        x_input = np.zeros((1, SEQ_LENGTH), dtype=np.int32)
        x_input[0, :len(tokens)] = tokens

        probs = model.predict(x_input, verbose=0)[0].astype(np.float64)

        # to stay within vocab cap
        probs[MAX_VOCAB:] = 0.0
        probs[0] = 0.0
        if UNK_ID < len(probs):
            probs[UNK_ID] = 0.0
        probs = probs / (np.sum(probs) + 1e-12)

        # temperature
        probs = np.log(probs + 1e-12) / temp
        probs = np.exp(probs)
        probs = probs / (np.sum(probs) + 1e-12)

        # top-k + top-p
        probs = filter_top_k_top_p(probs, top_k=top_k, top_p=top_p)

        next_id = np.random.choice(len(probs), p=probs)
        next_word = index_word.get(next_id, None)
        if not next_word:
            continue

        generated += " " + next_word

        if next_word in [".", "!", "?"] and i > 50:
            break

    return generated

# Demo
print("Model Ready for Demonstration")
print("Type 'exit' to stop.\n")

while True:
    user_input = input("Enter prompt words: ").strip()

    if user_input.lower() in ["exit"]:
        print("Exiting generation.")
        break

    if len(user_input) == 0:
        print("Please enter some text: \n")
        continue

    print("\nGenerated Agatha Christie Writing Style Output:\n")
    print(generate_poirot_text(user_input))

Mounted at /content/drive
Sample text:
 the adventure of " the western star " ii the tragedy at marsdon manor iii the adventure of the cheap flat iv the mystery of hunter's lodge v the million dollar bond robbery vi the adventure of the egyptian tomb vii jewel robbery at the grand metropolitan viii the kidnapped prime minister ix the disappearance of mr . davenheim x the adventure of the italian nobleman xi the case of the missing will poirot investigates poirot investigates i the adventure of " the western star " i was standing at th 

Saved tokenizer /content/drive/MyDrive/DLLM/tokenizer.pkl
Vocab cap: 10000
First 20 tokens: [5, 756, 9, 4, 5, 844, 396, 4, 845, 5, 1135, 28, 846, 460, 2134, 5, 756, 9, 5, 967]
Total sequences: (66527, 41)
X shape: (66527, 40) y shape: (66527,)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 40, 200)        │     2,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 40, 256)        │       467,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 40, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 256)            │       525,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10000)          │     2,570,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,563,280 (21.22 MB)

 Trainable params: 5,563,280 (21.22 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
468/468 ━━━━━━━━━━━━━━━━━━━━ 16s 24ms/step - accuracy: 0.0569 - loss: 6.7206 - val_accuracy: 0.0694 - val_loss: 6.2775
Epoch 2/30
468/468 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.1110 - loss: 5.8098 - val_accuracy: 0.1329 - val_loss: 5.9824
Epoch 3/30
468/468 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.1507 - loss: 5.3996 - val_accuracy: 0.1485 - val_loss: 5.9080
Epoch 4/30
468/468 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.1646 - loss: 5.1812 - val_accuracy: 0.1332 - val_loss: 6.0293
Epoch 5/30
468/468 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.1597 - loss: 5.3018 - val_accuracy: 0.1506 - val_loss: 5.8907
Epoch 6/30
468/468 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.1787 - loss: 5.0215 - val_accuracy: 0.1556 - val_loss: 5.9039
Epoch 7/30
468/468 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.1898 - loss: 4.8420 - val_accuracy: 0.1602 - val_loss: 5.9037
Epoch 8/30
468/468 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.1999 - loss: 4.6898 - val_accurac